# MedAI Lab - entraînement sur Colab

Prototype de recherche. Ne pas utiliser pour un diagnostic ou une décision clinique.

Ce notebook ne contient aucune logique métier : il clone le dépôt, installe les dépendances figées, lance `python -m medai_ml run` avec une configuration du dépôt, puis sauvegarde le rapport et les poids sur Google Drive.

Procédure : Exécution > Modifier le type d'exécution > GPU (T4) ; puis Exécuter tout. Si pip demande de redémarrer l'exécution, redémarre-la et relance à partir de la cellule 3.

In [ ]:
# 1. Matériel disponible (le GPU gratuit n'est pas garanti ; sans GPU, l'entraînement est très lent)
!nvidia-smi -L || echo 'Pas de GPU'

In [ ]:
# 2. Dépôt (public) à la version exacte à évaluer
REPO = 'https://github.com/AbdGhni/medai-lab.git'
REF = 'main'   # remplacer par un commit ou un tag pour une exécution de référence
!rm -rf /content/medai-lab
!git clone -q {REPO} /content/medai-lab
%cd /content/medai-lab
!git checkout -q {REF}
!git rev-parse HEAD
# Le rapport enregistre le commit et l'état du dépôt ; ce doit être vide pour un résultat de référence :
!git status --porcelain

In [ ]:
# 3. Dépendances figées (torch et torchvision viennent de Colab)
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 4. Configuration à exécuter (voir ml/configs/)
CONFIG = 'ml/configs/resnet18.yaml'
!cat {CONFIG}

In [ ]:
# 5. Entraînement + évaluation selon docs/EVAL_PROTOCOL.md (plusieurs seeds ; peut durer longtemps)
!python -m medai_ml run --config {CONFIG} --reports /content/out/reports --artifacts /content/out/artifacts

In [ ]:
# 6. Sauvegarde sur Google Drive (la session Colab est éphémère)
from google.colab import drive
import shutil, pathlib
drive.mount('/content/drive')
dest = pathlib.Path('/content/drive/MyDrive/medai-lab')
dest.mkdir(parents=True, exist_ok=True)
for sub in ('reports', 'artifacts'):
    src = pathlib.Path('/content/out') / sub
    if src.exists():
        shutil.copytree(src, dest / sub, dirs_exist_ok=True)
print('Copié dans', dest)
!ls -R /content/drive/MyDrive/medai-lab/reports | head -20

In [ ]:
# 7. (Option) Télécharger directement l'archive du rapport
import shutil
from google.colab import files
shutil.make_archive('/content/medai_reports', 'zip', '/content/out/reports')
files.download('/content/medai_reports.zip')